# Does a PCA rotation matter? XGBoost, LightGBM and TabM (IEEE-CIS fraud)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q lightgbm tabm

In [4]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn.functional as F
from tabm import TabM

warnings.filterwarnings('ignore')

QUICK = False          # True: 150k-row subsample, to test that everything runs
SPLIT = 'random'       # 'random' (stratified) or 'time' (chronological)
SEEDS = [0]            # e.g. [0, 1, 2] for mean +/- std
ARMS = ['scaled', 'pca']
MODELS = ['xgboost', 'lightgbm', 'tabm']
MAX_MISSING = 0.9      # drop numeric cols with > this fraction missing (train)
MIN_CAT_COUNT = 20     # rarer categories share the 'unknown' code
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# shared by the three tree models
GBDT = dict(lr=0.05, max_rounds=5000, patience=100, subsample=0.8, colsample=0.8)
XGB_DEPTH = 6
LGB_LEAVES = 63

TABM_CFG = dict(k=32, n_blocks=3, d_block=256, dropout=0.1,
                lr=2e-3, weight_decay=3e-4, batch_size=2048, max_epochs=100, patience=5)
print('device:', DEVICE)



device: cuda


## Load data

In [5]:
data_dir = '/content/drive/MyDrive/ieee-fraud-detection'
if not os.path.exists(os.path.join(data_dir, 'train_transaction.csv')):
    raise FileNotFoundError(f'train_transaction.csv not found in {data_dir}')

tx = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
idn_path = os.path.join(data_dir, 'train_identity.csv')
df = tx.merge(pd.read_csv(idn_path), on='TransactionID', how='left') if os.path.exists(idn_path) else tx
del tx

if QUICK:
    df = df.sample(150_000, random_state=0)
df = df.sort_values('TransactionDT').reset_index(drop=True)   # row order is now chronological

y = df['isFraud'].to_numpy(dtype=np.float32)
X = df.drop(columns=['isFraud', 'TransactionID', 'TransactionDT'])
del df

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]
X_num = X[num_cols].astype(np.float32)
X_cat = X[cat_cols]
del X

print(f'rows: {len(y):,} | fraud rate: {y.mean():.4%} | numeric cols: {len(num_cols)} | categorical cols: {len(cat_cols)}')

rows: 590,540 | fraud rate: 3.4990% | numeric cols: 400 | categorical cols: 31


## Split and preprocessing
Everything that learns from data (imputation medians, scaler, PCA, category maps) is fit on the train rows only.

In [6]:
def make_split(n, y, seed):
    idx = np.arange(n)
    if SPLIT == 'time':   # rows are already sorted by TransactionDT
        a, b = int(0.6 * n), int(0.8 * n)
        return idx[:a], idx[a:b], idx[b:]
    tr, tmp = train_test_split(idx, test_size=0.4, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=y[tmp], random_state=seed)
    return tr, va, te


def base_preprocess(tr, seed):
    # numeric: drop sparse columns, median-impute, quantile-scale
    miss = X_num.iloc[tr].isna().mean()
    keep = miss.index[miss <= MAX_MISSING]
    med = X_num.iloc[tr][keep].median()
    A = X_num[keep].fillna(med).to_numpy(dtype=np.float32)
    sc = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                             subsample=200_000, random_state=seed).fit(A[tr])
    A = sc.transform(A).astype(np.float32)

    # categorical: ordinal codes, 0 = unknown / rare / missing
    C = np.zeros((len(X_cat), len(cat_cols)), dtype=np.int64)
    cards = []
    for j, c in enumerate(cat_cols):
        s = X_cat[c]
        vc = s.iloc[tr].value_counts()
        kept = vc.index[vc >= MIN_CAT_COUNT]
        mapping = {v: i + 1 for i, v in enumerate(kept)}
        C[:, j] = s.map(mapping).fillna(0).astype(np.int64).to_numpy()
        cards.append(len(kept) + 1)
    return dict(num=A, cat=C, cards=cards)


def apply_arm(P, tr, arm, seed):
    if arm == 'scaled':
        return P
    pca = PCA(n_components=None, svd_solver='full', random_state=seed).fit(P['num'][tr])  # full-rank rotation, no whitening
    return dict(num=pca.transform(P['num']).astype(np.float32), cat=P['cat'], cards=P['cards'])

## Models



In [7]:
def stack(P):
    return np.hstack([P['num'], P['cat'].astype(np.float32)])


def run_xgboost(P, tr, va, te, seed):
    Z = stack(P)
    clf = xgb.XGBClassifier(objective='binary:logistic',
                            eval_metric=['logloss', 'aucpr'],   # early stopping uses the last one (aucpr)
                            tree_method='hist', device=DEVICE, random_state=seed, n_jobs=-1,
                            max_depth=XGB_DEPTH, learning_rate=GBDT['lr'], n_estimators=GBDT['max_rounds'],
                            early_stopping_rounds=GBDT['patience'],
                            subsample=GBDT['subsample'], colsample_bytree=GBDT['colsample'])
    clf.fit(Z[tr], y[tr], eval_set=[(Z[va], y[va])], verbose=100)
    best = int(clf.best_iteration)
    print(f'best iteration: {best}')
    return clf.predict_proba(Z[te])[:, 1], dict(best_iter=best, capped=best >= GBDT['max_rounds'] - 1)


def run_lightgbm(P, tr, va, te, seed):
    Z = stack(P)
    clf = lgb.LGBMClassifier(objective='binary', n_estimators=GBDT['max_rounds'], learning_rate=GBDT['lr'],
                             num_leaves=LGB_LEAVES, subsample=GBDT['subsample'], subsample_freq=1,
                             colsample_bytree=GBDT['colsample'], random_state=seed, n_jobs=-1, verbose=-1)
    # eval_metric is listed first; early stopping watches only that first metric (PR-AUC)
    clf.fit(Z[tr], y[tr], eval_set=[(Z[va], y[va])], eval_metric=['average_precision'],
            callbacks=[lgb.early_stopping(GBDT['patience'], first_metric_only=True, verbose=False),
                       lgb.log_evaluation(100)])
    best = int(clf.best_iteration_)
    print(f'best iteration: {best}')
    return clf.predict_proba(Z[te])[:, 1], dict(best_iter=best, capped=best >= GBDT['max_rounds'] - 1)


def run_tabm(P, tr, va, te, seed):
    cfg = TABM_CFG
    K = cfg['k']
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    dev = torch.device(DEVICE)

    Xn = torch.as_tensor(P['num'], device=dev)
    Xc = torch.as_tensor(P['cat'], device=dev) if P['cards'] else None
    Y = torch.as_tensor(y, device=dev)

    model = TabM.make(n_num_features=Xn.shape[1], cat_cardinalities=P['cards'] or None, d_out=1,
                      k=K, n_blocks=cfg['n_blocks'], d_block=cfg['d_block'], dropout=cfg['dropout']).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    def forward(b):
        return model(Xn[b], Xc[b] if Xc is not None else None).squeeze(-1)   # (batch, K)

    @torch.no_grad()
    def predict(idx):
        model.eval()
        out = []
        for i in range(0, len(idx), 8192):
            b = torch.as_tensor(idx[i:i + 8192], device=dev)
            out.append(torch.sigmoid(forward(b)).mean(1).float().cpu())      # average the K members' probabilities
        return torch.cat(out).numpy()

    best, bad, best_state, best_ep, capped = -1.0, 0, None, 0, True
    for epoch in range(1, cfg['max_epochs'] + 1):
        t0 = time.time()
        model.train()
        perm = torch.as_tensor(rng.permutation(tr), device=dev)
        total = torch.zeros((), device=dev)
        for i in range(0, len(perm), cfg['batch_size']):
            b = perm[i:i + cfg['batch_size']]
            loss = F.binary_cross_entropy_with_logits(forward(b), Y[b, None].expand(-1, K))  # per-member loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(b)
        pv = predict(va)
        ap = average_precision_score(y[va], pv)
        vl = log_loss(y[va], np.clip(pv, 1e-7, 1 - 1e-7))
        improved = ap > best
        mark = '  *' if improved else ''
        print(f'epoch {epoch:2d} | train loss {total.item() / len(tr):.5f} | val logloss {vl:.5f} '
              f'| val PR-AUC {ap:.4f}{mark} | {time.time() - t0:.0f}s')
        if improved:
            best, bad, best_ep = ap, 0, epoch
            best_state = {n: v.detach().clone() for n, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg['patience']:
                print(f'early stopping at epoch {epoch} (best val PR-AUC {best:.4f})')
                capped = False
                break
    model.load_state_dict(best_state)
    return predict(te), dict(best_iter=best_ep, capped=capped)


RUNNERS = dict(xgboost=run_xgboost, lightgbm=run_lightgbm, tabm=run_tabm)

## Experiment

In [8]:
rows = []
out_csv = os.path.join(data_dir, 'rotation_results.csv')

for seed in SEEDS:
    tr, va, te = make_split(len(y), y, seed)
    base = base_preprocess(tr, seed)
    print(f'\n================ seed {seed} | split: {SPLIT} ================')
    print(f'train/val/test: {len(tr):,}/{len(va):,}/{len(te):,} | test frauds: {int(y[te].sum())}')
    for arm in ARMS:
        P = apply_arm(base, tr, arm, seed)
        print(f'\n######## arm: {arm} | numeric features: {P["num"].shape[1]} ########')
        for name in MODELS:
            print(f'\n===== seed {seed} | {arm} | {name} =====')
            t0 = time.time()
            p, info = RUNNERS[name](P, tr, va, te, seed)
            rows.append(dict(seed=seed, split=SPLIT, arm=arm, model=name,
                             pr_auc=average_precision_score(y[te], p),
                             roc_auc=roc_auc_score(y[te], p),
                             secs=round(time.time() - t0), **info))
            print(f'test PR-AUC {rows[-1]["pr_auc"]:.4f} | ROC-AUC {rows[-1]["roc_auc"]:.4f}')
            pd.DataFrame(rows).to_csv(out_csv, index=False)

res = pd.DataFrame(rows)


================ seed 0 | split: random ================
train/val/test: 354,324/118,108/118,108 | test frauds: 4132

######## arm: scaled | numeric features: 390 ########

===== seed 0 | scaled | xgboost =====
[0]	validation_0-logloss:0.14023	validation_0-aucpr:0.38854
[100]	validation_0-logloss:0.08540	validation_0-aucpr:0.60767
[200]	validation_0-logloss:0.07907	validation_0-aucpr:0.65033
[300]	validation_0-logloss:0.07527	validation_0-aucpr:0.67659
[400]	validation_0-logloss:0.07237	validation_0-aucpr:0.69633
[500]	validation_0-logloss:0.06970	validation_0-aucpr:0.71422
[600]	validation_0-logloss:0.06776	validation_0-aucpr:0.72668
[700]	validation_0-logloss:0.06590	validation_0-aucpr:0.73832
[800]	validation_0-logloss:0.06432	validation_0-aucpr:0.74820
[900]	validation_0-logloss:0.06299	validation_0-aucpr:0.75628
[1000]	validation_0-logloss:0.06165	validation_0-aucpr:0.76425
[1100]	validation_0-logloss:0.06058	validation_0-aucpr:0.77048
[1200]	validation_0-logloss:0.05972	validati

## Summary

In [9]:
pd.options.display.float_format = '{:.4f}'.format

print('Test PR-AUC (mean / std over seeds):')
print(res.groupby(['model', 'arm'])['pr_auc'].agg(['mean', 'std', 'count']).to_string())

w = res.pivot_table(index=['seed', 'model'], columns='arm', values='pr_auc')
eff = (w['pca'] - w['scaled']).groupby('model').agg(['mean', 'std'])
print('\nRotation effect on PR-AUC (pca - scaled):')
print(eff.to_string())

wm = res.pivot_table(index='seed', columns=['model', 'arm'], values='pr_auc')
print('\nTabM minus each tree model (positive = TabM ahead):')
for tree in [m for m in ('xgboost', 'lightgbm') if m in MODELS]:
    g_s = wm[('tabm', 'scaled')] - wm[(tree, 'scaled')]
    g_p = wm[('tabm', 'pca')] - wm[(tree, 'pca')]
    print(f'  tabm - {tree:9s}: scaled {g_s.mean():+.4f} | pca {g_p.mean():+.4f} | shift {(g_p - g_s).mean():+.4f}')

capped = res[res['capped']]
if len(capped):
    print('\nWARNING: these runs hit their round/epoch limit, so their scores are lower bounds:')
    print(capped[['seed', 'arm', 'model', 'best_iter']].to_string(index=False))
else:
    print('\nAll runs stopped early (none hit the round/epoch limit).')

Test PR-AUC (mean / std over seeds):
                  mean  std  count
model    arm                      
lightgbm pca    0.6748  NaN      1
         scaled 0.8501  NaN      1
tabm     pca    0.6953  NaN      1
         scaled 0.7229  NaN      1
xgboost  pca    0.6610  NaN      1
         scaled 0.8305  NaN      1

Rotation effect on PR-AUC (pca - scaled):
            mean  std
model                
lightgbm -0.1753  NaN
tabm     -0.0276  NaN
xgboost  -0.1695  NaN

TabM minus each tree model (positive = TabM ahead):
  tabm - xgboost  : scaled -0.1076 | pca +0.0343 | shift +0.1420
  tabm - lightgbm : scaled -0.1272 | pca +0.0205 | shift +0.1477

All runs stopped early (none hit the round/epoch limit).
